# Playlist Generator — Text Prompt → k-NN

Generates a playlist from a text prompt using pre-computed CLAP audio embeddings.

**Prerequisite:** run `compute_embeddings.ipynb` at least once to build `data/embeddings.npz`.

Pipeline:
1. Load cached audio embeddings from disk (fast, no GPU required for this step)
2. Encode the text prompt with CLAP
3. Cosine k-NN → ranked playlist

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch

if "" not in sys.path:
    sys.path.insert(0, "")

import utils

In [ ]:
K               = 20    # tracks per playlist

EMB_PATH        = Path("data/embeddings.npz") 

CHECKPOINT_URL  = utils.DEFAULT_CHECKPOINT_URL
CHECKPOINT_PATH = utils.DEFAULT_CHECKPOINT_PATH

DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"K      : {K}")
print(f"Device : {DEVICE}")

Prompt : 'slow chill hip hop'
K      : 20
Device : cuda


In [ ]:
audio_paths, audio_embs = utils.load_embeddings(EMB_PATH)

Loaded 7994 embeddings from data/embeddings.npz  shape=(7994, 512)


In [ ]:
# load
model = utils.load_clap_model(CHECKPOINT_PATH, CHECKPOINT_URL, device=DEVICE)

Checkpoint already present: checkpoints/music_audioset_epoch_15_esc_90.14.pt


/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint checkpoints/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.bl

In [ ]:
# Prompt
PROMPT          = "Techno music"
text_emb = model.get_text_embedding([PROMPT], use_tensor=False)
text_emb = utils.l2_normalize(np.asarray(text_emb, dtype=np.float32))[0]
print(f"Text embedding shape: {text_emb.shape}")

Text embedding shape: (512,)


In [9]:
# knn
indices, scores = utils.knn_search(text_emb, audio_embs, k=K)

print(f"\nPlaylist for prompt: {PROMPT!r}")
print(f"{'Rank':<5} {'Score':>6}  Track")
print("-" * 70)
for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
    track_name = Path(audio_paths[idx]).name
    print(f"{rank:<5} {score:>6.4f}  {track_name}")


Playlist for prompt: 'Gangsta american rap'
Rank   Score  Track
----------------------------------------------------------------------
1     0.5127  013749.mp3
2     0.5027  011917.mp3
3     0.5024  071228.mp3
4     0.4966  148028.mp3
5     0.4809  142093.mp3
6     0.4795  013747.mp3
7     0.4696  142087.mp3
8     0.4681  149690.mp3
9     0.4669  149689.mp3
10    0.4658  080293.mp3
11    0.4649  123147.mp3
12    0.4644  098576.mp3
13    0.4598  144179.mp3
14    0.4564  132964.mp3
15    0.4560  026034.mp3
16    0.4551  129915.mp3
17    0.4535  007491.mp3
18    0.4526  096678.mp3
19    0.4484  007489.mp3
20    0.4454  128494.mp3


In [10]:
# ── Audio preview — top 5 tracks ─────────────────────────────────────────────
from IPython.display import display, Audio, HTML

TOP_PREVIEW = 5

display(HTML(f"<h3> Top {TOP_PREVIEW} preview <em>{PROMPT}</em></h3>"))

for rank, (idx, score) in enumerate(zip(indices[:TOP_PREVIEW], scores[:TOP_PREVIEW]), start=1):
    path = audio_paths[idx]
    name = Path(path).name
    display(HTML(f"<b>#{rank} &nbsp; {name} &nbsp; <span style='color:gray'>score: {score:.4f}</span></b>"))
    display(Audio(path, autoplay=False))